# Validator behaviour, supervised: who stakes again within a week?

| Field | Value |
| --- | --- |
| Question | At the end of a staking period, can the node's next move be predicted from what was known at that moment? |
| Unit | One row per validator staking period that ended before the lockbox, with features as of its end |
| Targets | `renewed`: a next period on the same node within 7 days. `outcome`: renew, return within 365 days, or leave. `next_duration`: the class of the next period's duration |
| Validation | Forward in time: six-month test windows from 2022-11-19 to the lockbox, each trained only on labels known when it starts |
| Benchmarks | `prior`, the training rate, and `history`, the node's own earlier renewal share shrunk toward it |
| Cut | Nothing from 2025-11-19 16:00 UTC onward is read |
| Library | `avalanche_research.behaviour`: `events`, `features`, `supervised`, `views` |
| Status | v0, unreviewed |

Restart the kernel and run all.

In [1]:
import json
import warnings
from pathlib import Path

import hvplot.pandas  # noqa: F401
import pandas as pd
import panel as pn
from IPython.display import Markdown, display

from avalanche_research import behaviour
from avalanche_research.behaviour import events as ev, features as ft, supervised as sv, views

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", message=".*generic' unit for NumPy timedelta", category=DeprecationWarning)
pn.extension("tabulator")
pd.set_option("display.max_columns", 40)

inputs = behaviour.load()
events = ev.decision_events(inputs.periods)
features = ft.build(events, inputs)
named = events.node_id.map(inputs.nodes.named_agent)
ft.registry_frame()

,name,family,description,source,kind
0,stake_log10,period,"log10 of the period's stake, AVAX",staking-periods.stake_avax,numeric
1,duration_days,period,"Length of the period, days",staking-periods.duration_days,numeric
2,fee_rate,period,"Declared delegation fee, 0.02 to 1",staking-periods.delegation_fee_rate,numeric
3,failed,period,"1 if the period earned no reward, the uptime rule",staking-periods.rewarded,numeric
4,reward_rate_annual,period,"Potential reward over stake, annualised over t...",staking-periods.potential_reward_avax,numeric
5,pending_start,period,1 if the period was accepted before it started,staking-periods.start_kind,numeric
6,at_stake_floor,period,"1 if the stake is the 2,000 AVAX minimum",staking-periods.stake_avax,numeric
7,fee_at_floor,period,1 if the fee is the 2% minimum,staking-periods.delegation_fee_rate,numeric
8,regime,period,Network upgrade regime when the period was acc...,staking-periods.regime,categorical
9,period_number,history,"This period's place in the node's sequence, fr...",staking-periods,numeric


## 1. No feature reads the future

Features are rebuilt from data cut at 2024-06-01 and compared with the full build for every decision before that date. Any difference would mean a feature used information from after the decision. The first version of `prior_mean_gap_days` failed exactly this check: its missing values marked the nodes that never came back.

In [2]:
cut = "2024-06-01"
early_inputs = behaviour.load(observed_through=cut)
early_events = ev.decision_events(early_inputs.periods, observed_through=cut)
early = ft.build(early_events, early_inputs).set_index(early_events.transaction_hash)
full = features.set_index(events.transaction_hash).loc[early.index]
numeric = [c for c in early.columns if c not in ft.CATEGORICAL]
differ = ((early[numeric] - full[numeric]).abs().gt(1e-9) | early[numeric].isna().ne(full[numeric].isna())).sum()
assert differ.sum() == 0, differ[differ > 0]
f"{len(early):,} decisions compared, no feature differs"

'30,046 decisions compared, no feature differs'

## 2. The label is the registered measure

`renewed` must be the `AM2` measure of the validation protocol, not a near copy. Its share over every ended period with a full week of follow-up equals the renewal helper's for 2021 to 2024 and for 2025 to the lockbox.

In [3]:
from avalanche_research import staking_renewal

measured = staking_renewal.renewal_events(inputs.periods[["transaction_hash", "node_id", "start_time", "end_time", "stake_avax"]])
rows = []
for start, end in (("2021-01-01", "2025-01-01"), ("2025-01-01", behaviour.LOCKBOX_START)):
    ours = events.end_time.ge(start) & events.end_time.lt(end) & events.renewed.notna() \
        & (events.end_time + pd.Timedelta(days=7)).lt(behaviour.LOCKBOX_START)
    reference = staking_renewal.renewal_summary(measured, start, end, behaviour.LOCKBOX_START)
    rows.append({"from": start, "to": str(end), "decisions": int(ours.sum()), "renewed": events.loc[ours, "renewed"].mean(),
                 "measure_endings": reference["eligible_ends"], "measure": reference["renewal_share"]})
check = pd.DataFrame(rows)
assert (check.decisions == check.measure_endings).all() and (check.renewed - check.measure).abs().max() < 1e-12
check

,from,to,decisions,renewed,measure_endings,measure
0,2021-01-01,2025-01-01,38149,0.801541,38149,0.801541
1,2025-01-01,2025-11-19 16:00:00,12808,0.778576,12808,0.778576


## 3. Seven-day renewal on every node, and on unnamed nodes

GoGoPool's minipools alone are about a quarter of all validator periods, and the operators named in the attribution file (GoGoPool, BENQI, holders of genesis allocations) follow their own rules rather than the population's. Each target is therefore evaluated twice: on every node, and on unnamed nodes only. The named operators are a separate regime, to be modelled as named actors.

In [4]:
unnamed = named.isna()
renewed_all = sv.run(events, features, "renewed")
renewed = sv.run(events, features, "renewed", keep=unnamed)
display(pd.DataFrame([{"fold": f.name, "train": len(f.train), "test": len(f.test)} for f in renewed.folds]).set_index("fold").T)
display(Markdown("**Every node**"), sv.summarise(renewed_all.scores).round(3))
display(Markdown("**Unnamed nodes**"), sv.summarise(renewed.scores).round(3))
views.panel(views.FoldScores(scores=renewed.scores, name="Scores, unnamed nodes"), ["metric"])

fold,2022-11,2023-05,2023-11,2024-05,2024-11,2025-05
train,14039,17251,20225,23524,26698,29819
test,3231,2952,3331,3158,3139,2994


**Every node**

,log_loss,brier,roc_auc,average_precision,accuracy,skill_vs_prior,last_fold_log_loss,last_fold_brier,last_fold_roc_auc,last_fold_average_precision,last_fold_accuracy,last_fold_skill_vs_prior
model,,,,,,,,,,,,
history,0.485,0.299,0.710,0.384,0.809,0.030,0.599,0.365,0.673,0.401,0.772,-0.090
prior,0.498,0.318,0.500,0.197,0.803,0.000,0.550,0.363,0.500,0.235,0.765,0.000
logistic,0.507,0.304,0.698,0.364,0.799,-0.020,0.519,0.337,0.671,0.400,0.765,0.055
boosted,0.533,0.329,0.667,0.346,0.778,-0.072,0.555,0.324,0.707,0.495,0.794,-0.009


**Unnamed nodes**

,log_loss,brier,roc_auc,average_precision,accuracy,skill_vs_prior,last_fold_log_loss,last_fold_brier,last_fold_roc_auc,last_fold_average_precision,last_fold_accuracy,last_fold_skill_vs_prior
model,,,,,,,,,,,,
boosted,0.444,0.285,0.767,0.475,0.797,0.146,0.419,0.272,0.808,0.536,0.800,0.201
history,0.485,0.306,0.743,0.444,0.795,0.069,0.512,0.318,0.729,0.452,0.792,0.021
logistic,0.497,0.314,0.737,0.437,0.784,0.041,0.461,0.298,0.758,0.467,0.794,0.119
prior,0.520,0.337,0.500,0.214,0.786,0.000,0.524,0.340,0.500,0.217,0.783,0.000


Row(sizing_mode='stretch_width')
    [0] Param(FoldScores, name='Scores, unnamed nodes', parameters=['metric'], show_name=False, widgets={'metric': {'sizing_mode':...}, width=260)
    [1] ParamMethod(method, _pane=HoloViews, defer_load=False)

## 4. Calibration on unnamed nodes

A model fed into a simulator must give probabilities that mean what they say. Each point is one probability bin over all test windows: on the diagonal, predicted renewal equals observed renewal.

In [5]:
tables = {model: sv.calibration(renewed.predictions, model) for model in ("boosted", "logistic", "history")}
views.panel(views.Calibration(tables=tables, name="Calibration"), ["model"])

Row(sizing_mode='stretch_width')
    [0] Param(Calibration, parameters=['model'], show_name=False, widgets={'model': {'sizing_mode': ...}, width=260)
    [1] ParamMethod(method, _pane=HoloViews, defer_load=False)

## 5. What the boosted model relies on

Permutation importance on the last test window before the lockbox, unnamed nodes: how much the log loss rises when one feature's values are shuffled. This is association within one reward regime, not the effect of changing that feature.

In [6]:
last = renewed.folds[-1]
boosted = sv.fit_on(sv.models()["boosted"], renewed.X, renewed.y, last.train)
importance = sv.importance(boosted, renewed.X.iloc[last.test], renewed.y.iloc[last.test])
display(importance.head(15).round(4))
views.importance_bars(importance)

,feature,family,importance,spread
0,period_number,history,0.0255,0.0032
1,prior_mean_gap_days,history,0.0216,0.0021
2,prev_gap_days,history,0.0204,0.0022
3,stake_log10,period,0.0106,0.0008
4,duration_days,period,0.0106,0.0007
5,failed,period,0.0091,0.0018
6,stake_share_of_network_bp,monetary,0.0083,0.0013
7,operator_nodes_to_date,operator,0.0063,0.0012
8,node_age_days,history,0.0031,0.0022
9,stake_vs_max_log,history,0.0022,0.0002


:Bars   [feature,family]   (importance)

## 6. Renew, return, or leave

The three-way outcome needs a year of follow-up, so its last test window ends a year before the lockbox.

In [7]:
outcome_all = sv.run(events, features, "outcome")
outcome = sv.run(events, features, "outcome", keep=unnamed)
display(outcome.y.value_counts(normalize=True).round(3).rename("share, unnamed").to_frame())
display(Markdown("**Every node**"), sv.summarise(outcome_all.scores).round(3))
display(Markdown("**Unnamed nodes**"), sv.summarise(outcome.scores).round(3))

,"share, unnamed"
outcome,
renew,0.799
leave,0.106
return,0.094


**Every node**

,log_loss,brier,roc_auc,accuracy,skill_vs_prior,last_fold_log_loss,last_fold_brier,last_fold_roc_auc,last_fold_accuracy,last_fold_skill_vs_prior
model,,,,,,,,,,
prior,0.616,0.322,0.500,0.816,0.000,0.616,0.326,0.500,0.810,0.000
boosted,0.908,0.501,0.653,0.648,-0.505,1.098,0.588,0.649,0.583,-0.783
logistic,0.921,0.474,0.696,0.680,-0.512,0.795,0.347,0.717,0.811,-0.291


**Unnamed nodes**

,log_loss,brier,roc_auc,accuracy,skill_vs_prior,last_fold_log_loss,last_fold_brier,last_fold_roc_auc,last_fold_accuracy,last_fold_skill_vs_prior
model,,,,,,,,,,
boosted,0.609,0.332,0.742,0.779,0.112,0.553,0.301,0.768,0.798,0.137
prior,0.686,0.366,0.500,0.783,0.000,0.641,0.339,0.500,0.802,0.000
logistic,0.968,0.467,0.711,0.696,-0.420,0.784,0.344,0.729,0.800,-0.222


## 7. The next duration

Among nodes that stake again within a year, the duration class of the next period. The benchmark here is `persistence`: the training frequency of each next class given the current period's class.

In [8]:
duration_all = sv.run(events, features, "next_duration")
duration = sv.run(events, features, "next_duration", keep=unnamed)
display(duration.y.value_counts(normalize=True).round(3).rename("share, unnamed").to_frame())
display(Markdown("**Every node**"), sv.summarise(duration_all.scores).round(3))
display(Markdown("**Unnamed nodes**"), sv.summarise(duration.scores).round(3))

,"share, unnamed"
next_duration_class,
3 to 6 weeks,0.453
up to 3 weeks,0.234
6 weeks to 3 months,0.205
3 to 6 months,0.061
over 6 months,0.046


**Every node**

,log_loss,brier,roc_auc,accuracy,skill_vs_prior,last_fold_log_loss,last_fold_brier,last_fold_roc_auc,last_fold_accuracy,last_fold_skill_vs_prior
model,,,,,,,,,,
persistence,0.717,0.353,0.893,0.821,0.464,0.610,0.292,0.921,0.889,0.563
boosted,0.887,0.483,0.879,0.619,0.334,0.677,0.382,0.922,0.729,0.515
prior,1.338,0.718,0.500,0.338,0.000,1.395,0.766,0.500,0.206,0.000
logistic,2.724,1.017,0.697,0.284,-1.017,3.878,1.348,0.732,0.090,-1.780


**Unnamed nodes**

,log_loss,brier,roc_auc,accuracy,skill_vs_prior,last_fold_log_loss,last_fold_brier,last_fold_roc_auc,last_fold_accuracy,last_fold_skill_vs_prior
model,,,,,,,,,,
persistence,0.906,0.459,0.842,0.706,0.329,0.864,0.434,0.864,0.729,0.361
boosted,1.019,0.521,0.848,0.620,0.247,0.848,0.456,0.881,0.675,0.373
prior,1.351,0.689,0.500,0.484,0.000,1.352,0.699,0.500,0.445,0.000
logistic,2.118,0.893,0.690,0.375,-0.563,2.716,1.133,0.718,0.215,-1.009


## 8. Receipts

Each target's result is written with its inputs, feature list and hash, folds, scores, scikit-learn version and seed, to `notebooks/results/behaviour-v0/`.

In [9]:
out = Path("results/behaviour-v0")
out.mkdir(parents=True, exist_ok=True)
for name, result in (("renewed-all", renewed_all), ("renewed-unnamed", renewed), ("outcome-all", outcome_all),
                     ("outcome-unnamed", outcome), ("next-duration-all", duration_all),
                     ("next-duration-unnamed", duration)):
    record = sv.receipt(inputs.receipt(), result.target, result.X.columns, result.folds, sv.summarise(result.scores))
    record["rows"] = "unnamed nodes" if name.endswith("unnamed") else "every node"
    (out / f"{name}.json").write_text(json.dumps(record, indent=2, sort_keys=True) + "\n")
sorted(p.name for p in out.iterdir())

['next-duration-all.json',
 'next-duration-unnamed.json',
 'outcome-all.json',
 'outcome-unnamed.json',
 'renewed-all.json',
 'renewed-unnamed.json']

## Findings

Written from the run above. Codes are `SF1` onward.

| Code | Finding |
| --- | --- |
| `SF1` | No feature reads the future: features built from data cut at 2024-06-01 equal the full build for all 30,046 earlier decisions. The first version failed this check. A missing `prior_mean_gap_days` marked nodes that never came back, and lifted the area under the curve from 0.70 to 0.94. The check now runs in this notebook and in the tests |
| `SF2` | On every node, no learned model beats the base rate for 7-day renewal over all test windows. The node's own record does slightly (skill 0.03). The named operators follow their own rules, and pooling them with everyone else breaks the fit |
| `SF3` | On unnamed nodes, gradient boosting beats both benchmarks: skill over the base rate 0.15 across windows and 0.20 in the last window before the lockbox (area under the curve 0.81), against 0.07 and 0.02 for the node's own record. It is calibrated within 0.03 above a predicted 0.55 and under-predicts renewal in the sparser bins below |
| `SF4` | What it relies on is the node's history (its period number and the gaps after earlier periods), then stake, duration, a missed-uptime period, its share of network stake, and how many nodes share its reward address. Market and monetary features add almost nothing within one reward regime, which is consistent with the response to rewards being unidentified in this history |
| `SF5` | Renew, return or leave: on unnamed nodes boosting beats the base rate (skill 0.11, last window 0.14). On every node it is worse than the base rate. Multinomial logistic regression fails on both, because it extrapolates across the shift to short periods at Durango |
| `SF6` | The next duration is mostly the current one. A transition table of next class given current class is right 82% of the time on every node (71% unnamed), and beats gradient boosting on log loss (0.72 against 0.89). Duration choice needs a model of the change, not of the level |
| `SF7` | For the lifecycle model, the candidate inputs are renewal probabilities by tenure and gap history for unnamed nodes, with the named operators as separate actors and duration as a transition table. None of this has been scored on the lockbox. That scoring, against persistence, is the Milestone 2 test and waits for Shawn's review of this branch |